**News Classification**
  


In [4]:
import pandas as pd
import re


import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             classification_report, confusion_matrix)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

**LOAD DATASET**

In [5]:

df = pd.read_csv("newsdata01.xls")
print(f"\n 1. Dataset loaded")
print(f"    Rows , columns    : {df.shape} articles")
print(f"    Columns : {df.columns.tolist()}")
print(f"\n    Category distribution:")
print(df["category"].value_counts().to_string())



 1. Dataset loaded
    Rows , columns    : (3608, 2) articles
    Columns : ['text', 'category']

    Category distribution:
category
EDUCATION    902
POLITICS     902
SPORTS       902
TECH         902


**PREPROCESSING**

In [6]:

stop_words = set(stopwords.words('english'))
stemmer    = PorterStemmer()
def preprocess(text):


    text = text.lower()


    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()


    words = word_tokenize(text)

    filtered_words = [word for word in words if word.lower() not in stop_words and len(word) > 2]


    stemmed_words = [stemmer.stem(word) for word in filtered_words]

    return " ".join(stemmed_words)
    print("\n 2. Preprocessing all articles ...")
df["clean_text"] = df["text"].apply(preprocess)

sample_sentence = df["text"][0]
sample_words    = word_tokenize(sample_sentence.lower())
sample_filtered = [w for w in sample_words if w.lower() not in stop_words]
sample_stemmed  = [stemmer.stem(w) for w in sample_filtered]

print("\nThe original sentence is:                  ", sample_sentence[:80], "...")
print("\nThe filtered words after removing stopwords:", sample_filtered[:10])
print("\nThe PorterStemmer words are:                ", sample_stemmed[:10])
print("\nThe stop words from the library are (first 10):", list(stop_words)[:10])




The original sentence is:                   Teachers Swarm Kentucky Capitol To Protest Pension Changes, School Budget Cuts “ ...

The filtered words after removing stopwords: ['teachers', 'swarm', 'kentucky', 'capitol', 'protest', 'pension', 'changes', ',', 'school', 'budget']

The PorterStemmer words are:                 ['teacher', 'swarm', 'kentucki', 'capitol', 'protest', 'pension', 'chang', ',', 'school', 'budget']

The stop words from the library are (first 10): ['did', 'were', 'who', 'during', 'has', 'then', 'by', 'down', 'her', 'been']


**TRAIN / TEST SPLIT**

In [7]:

X = df["clean_text"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n 3.  Train / Test split (80% / 20%)")
print(f"    Training   : {len(X_train)}")
print(f"    Testing    : {len(X_test)}")


 3.  Train / Test split (80% / 20%)
    Training   : 2886
    Testing    : 722


**FEATURE EXTRACTION — TF-IDF**

In [8]:
print("\n 4.  Feature extraction using TF-IDF ...")
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"    Feature matrix shape : {X_train_tfidf.shape}")
print(f"    (articles × word features)")


 4.  Feature extraction using TF-IDF ...
    Feature matrix shape : (2886, 5000)
    (articles × word features)


**MODEL — LOGISTIC REGRESSION**

In [9]:
print("\n 5. Training Logistic Regression model ...")
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)


 5. Training Logistic Regression model ...


LogisticRegression(max_iter=1000, random_state=42)

**EVALUATION**

In [10]:

y_pred = model.predict(X_test_tfidf)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall    = recall_score(y_test, y_pred, average="weighted")
f1        = f1_score(y_test, y_pred, average="weighted")

print("\n 6. Evaluation Results")
print("-" * 40)
print(f"    Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"    Precision : {precision:.4f}")
print(f"    Recall    : {recall:.4f}")
print(f"    F1-Score  : {f1:.4f}")
print("\n    Detailed Report (per category):")
print(classification_report(y_test, y_pred))
print("    Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
cm_df = pd.DataFrame(cm, index=model.classes_, columns=model.classes_)
print(cm_df.to_string())
print("    (Rows = Actual label, Columns = Predicted label)")



 6. Evaluation Results
----------------------------------------
    Accuracy  : 0.8767  (87.67%)
    Precision : 0.8773
    Recall    : 0.8767
    F1-Score  : 0.8769

    Detailed Report (per category):
              precision    recall  f1-score   support

   EDUCATION       0.90      0.87      0.88       175
    POLITICS       0.82      0.85      0.84       172
      SPORTS       0.89      0.90      0.89       186
        TECH       0.89      0.89      0.89       189

    accuracy                           0.88       722
   macro avg       0.88      0.88      0.88       722
weighted avg       0.88      0.88      0.88       722

    Confusion Matrix:
           EDUCATION  POLITICS  SPORTS  TECH
EDUCATION        152        14       7     2
POLITICS           8       146       5    13
SPORTS             5         9     167     5
TECH               4         8       9   168
    (Rows = Actual label, Columns = Predicted label)


**PREDICT NEW ARTICLES**

In [12]:

print("\n 7. Predict new articles (demo)")
print("-" * 40)

sample_articles = [
    "The team scored two goals in first round.",
    " Apple had a new version of ios24",
    "The government announced new tax policies for the upcoming fiscal year.",
    "you had an english exam today did you ",
]

for article in sample_articles:
    clean = preprocess(article)
    vec   = tfidf.transform([clean])
    pred  = model.predict(vec)[0]
    print(f"  TEXT : {article}...")
    print(f"  PRED : {pred}\n")



 7. Predict new articles (demo)
----------------------------------------
  TEXT : The team scored two goals in first round....
  PRED : SPORTS

  TEXT :  Apple had a new version of ios24...
  PRED : TECH

  TEXT : The government announced new tax policies for the upcoming fiscal year....
  PRED : POLITICS

  TEXT : you had an english exam today did you ...
  PRED : EDUCATION

